# Sistemas de Recomendação com Avaliações (Ratings)

Este notebook utiliza os arquivos `movies.csv` e `ratings.csv` para criar dois modelos avançados de recomendação:
1. **Best Seller Ponderado (Weighted Rating - Fórmula IMDb)**: Combina a média de notas com o volume de avaliações para criar um ranking justo de populares.
2. **Similaridade por Filtragem Colaborativa Item-Item (Cosine Similarity)**: Calcula a similaridade entre filmes com base nos padrões reais de avaliação dos usuários.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.metrics.pairwise import cosine_similarity

# 1. Carregar os conjuntos de dados
movies = pd.read_csv('movies.csv')
ratings = pd.read_csv('ratings.csv')

print("Quantidade de Filmes:", movies.shape[0])
print("Quantidade de Avaliações:", ratings.shape[0])
ratings.head()

In [ ]:
# Agrupar avaliações por filme
stats = ratings.groupby('movieId').agg(
    vote_count=('rating', 'count'),
    vote_average=('rating', 'mean')
).reset_index()

# Unir com o dataframe de filmes
df_movies = pd.merge(movies, stats, on='movieId', how='left')
df_movies['vote_count'] = df_movies['vote_count'].fillna(0)
df_movies['vote_average'] = df_movies['vote_average'].fillna(0)

df_movies.head()

## 1. Sistema Best Seller (Weighted Rating - IMDb)

Utiliza a fórmula ponderada:
$$\text{Weighted Rating (WR)} = \left(\frac{v}{v + m} \cdot R\right) + \left(\frac{m}{v + m} \cdot C\right)$$
onde:
- $v$: número de avaliações do filme.
- $m$: quantidade mínima de votos necessária (ex: percentil 70).
- $R$: média das notas do filme.
- $C$: média geral de notas de todo o catálogo.

In [ ]:
C = df_movies['vote_average'].mean()
m = df_movies['vote_count'].quantile(0.70)

def weighted_rating(x, m=m, C=C):
    v = x['vote_count']
    R = x['vote_average']
    if v + m == 0:
        return 0
    return (v / (v + m) * R) + (m / (v + m) * C)

df_movies['weighted_score'] = df_movies.apply(weighted_rating, axis=1)

def obter_best_sellers(genero=None, top_n=10):
    df_filtered = df_movies.copy()
    if genero:
        df_filtered = df_filtered[df_filtered['genres'].str.contains(genero, case=False, na=False)]
    
    resultado = df_filtered.sort_values('weighted_score', ascending=False)
    cols = ['movieId', 'title', 'genres', 'vote_count', 'vote_average', 'weighted_score']
    return resultado[cols].head(top_n)

# Teste: Top 10 Best Sellers Globais
print("=== TOP 10 BEST SELLERS GLOBAIS ===")
obter_best_sellers(top_n=10)

## 2. Sistema de Similaridade por Filtragem Colaborativa Item-Item

Cria a matriz **Usuário-Item** e calcula o ângulo de similaridade (Cosine Similarity) entre os vetores de avaliações dos filmes.

In [ ]:
# Criar Matriz Usuário-Item (movieId nas linhas, userId nas colunas)
user_item_matrix = ratings.pivot(index='movieId', columns='userId', values='rating').fillna(0)

# Calcular similaridade entre filmes com base nas notas dos usuários
item_sim_matrix = cosine_similarity(user_item_matrix)
item_sim_df = pd.DataFrame(item_sim_matrix, index=user_item_matrix.index, columns=user_item_matrix.index)

def recomendar_por_similaridade_ratings(titulo, top_n=5):
    """
    Recebe o título de um filme e encontra outros filmes avaliados de forma semelhante pelos mesmos usuários.
    """
    matches = df_movies[df_movies['title'].str.contains(titulo, case=False, na=False)]
    if matches.empty:
        print(f"Filme '{titulo}' não encontrado.")
        return None
    
    target_movieId = matches.iloc[0]['movieId']
    target_title = matches.iloc[0]['title']
    
    if target_movieId not in item_sim_df.index:
        print(f"Filme '{target_title}' não tem avaliações suficientes.")
        return None
    
    sim_scores = item_sim_df[target_movieId].sort_values(ascending=False)[1:top_n+1]
    
    recs = df_movies[df_movies['movieId'].isin(sim_scores.index)].copy()
    recs['similarity_score'] = recs['movieId'].map(sim_scores)
    recs = recs.sort_values('similarity_score', ascending=False)
    
    cols = ['movieId', 'title', 'genres', 'vote_count', 'vote_average', 'similarity_score']
    print(f"=== Recomendações Colaborativas para: '{target_title}' ===")
    return recs[cols]

# Teste: Recomendação para 'Toy Story'
recomendar_por_similaridade_ratings('Toy Story', top_n=5)

In [ ]:
# Visualização das notas mais frequentes
plt.figure(figsize=(8, 4))
sns.countplot(data=ratings, x='rating', palette='viridis')
plt.title('Distribuição das Avaliações dos Usuários')
plt.xlabel('Nota (Rating)')
plt.ylabel('Quantidade de Avaliações')
plt.show()